# Generating the Solutions for the Selected Problems
Here, I want to use **Qwen2.5-Coder-14B-Instruct** to generate the solutions for the selected problems. Each of them will be written in Python, C#, and Typescript. The model is available on [Hugging Face](https://huggingface.co/Qwen/Qwen2.5-Coder-14B-Instruct).

In [1]:
# !pip install transformers
# !pip install collections 

In [2]:
import re
import os
import math
import torch
import pandas as pd
from tqdm import tqdm
from enum import Enum
from collections import defaultdict
from transformers import AutoTokenizer, AutoModelForCausalLM

## Selected Problems Loading
Firstly, I need to load the selected problems dataframe.

In [3]:
csv_file = '/datasets/selected_problems.csv'
csv_full_path = os.getcwd()+csv_file
if os.path.exists(csv_full_path):
    selected_problems = pd.read_csv(csv_full_path)
else:
    selected_problems = None
    print("File not found.")

In [4]:
selected_problems["solution"] = ''
selected_problems.shape

(30, 5)

In [5]:
selected_problems.head()

,title,description,difficulty,source,solution
0,Sports Rank,Sports leagues rank their participants (teams ...,Medium,Kattis,
1,Solidus Fraction,A fraction written with a solidus (/) can be w...,Medium,Kattis,
2,Breakout,A variant of the popular video game breakout u...,Medium,Kattis,
3,Passport Dates,Konráð has managed to obtain a big pile of Ice...,Easy,Kattis,
4,Weather Forecast,One of the most important things you can do in...,Hard,Kattis,


## Loading Qwen2.5-Coder-14B-Instruct Model

In [6]:
class Language(Enum):
    PYTHON = "python"
    TYPESCRIPT = "typescript"
    C_SHARP = "csharp"

In [7]:
model_name = "Qwen/Qwen2.5-Coder-14B-Instruct"
# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# print(device)

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)#.to(device)

2025-04-25 08:17:36.432808: I tensorflow/core/util/port.cc:111] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-04-25 08:17:36.471804: E tensorflow/compiler/xla/stream_executor/cuda/cuda_dnn.cc:9342] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-04-25 08:17:36.471824: E tensorflow/compiler/xla/stream_executor/cuda/cuda_fft.cc:609] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-04-25 08:17:36.471851: E tensorflow/compiler/xla/stream_executor/cuda/cuda_blas.cc:1518] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-04-25 08:17:36.479773: I tensorflow/core/platform/cpu_feature_g

Loading checkpoint shards:   0%|          | 0/6 [00:00<?, ?it/s]

## Code Generation - Prompt Engineering & Token Probabilties

In [8]:
c_sharp_prompt = "You are a C# developer. Please, solve this problem avoid using any built-in functions or libraries that simplify algorithmic steps:\n"
python_prompt = "You are a Python developer. Please, solve this problem avoid using any built-in functions or libraries that simplify algorithmic steps:\n"
typescript_prompt = "You are a Typescript developer. Please, solve this problem avoid using any built-in functions or libraries that simplify algorithmic steps:\n"

In [9]:
extra_instructions = "\nDo not add any extra text, explanations or comments."

In [10]:
def get_code_from_text(text, programming_language):
    code_match = re.search(r'```(.*?)```', text, re.DOTALL)
    if code_match:
        extracted_code = code_match.group(1).strip().replace(programming_language.value+'\n', '')
        return extracted_code
    else:
        return "No code found in the generated output"

In [11]:
def get_normalized_prob_line_by_line(logits, generated_tokens, context_length, generated_code, comment_symbol, programming_language, tokenizer):
    token_probs = defaultdict(list)
    
    code_token = "``"
    code_started = False
    
    current_line = 0
    lines = defaultdict(list)
    current_line_text = ""
    comment_started = False

    for i, token_logits in enumerate(logits): 
        token_id = generated_tokens[i + context_length].item()
        decoded_token = tokenizer.decode(token_id, skip_special_tokens=True)
        current_line_text += decoded_token
        if not code_started and code_token not in decoded_token: # the code not started
            continue

        if not code_started and code_token in decoded_token: # the code started
            code_started = True
            continue
        
        if code_started and code_token in decoded_token: # the code finished
            break
        
        if comment_symbol in decoded_token:
            comment_started = True
        
        if not comment_started:
            token_prob = torch.softmax(token_logits, dim=-1)[0][token_id].item()
            token_probs[current_line].append(token_prob)

        if '\n' in decoded_token:
            lines[current_line] = current_line_text
            current_line_text = ""
            current_line += 1
            token_probs[current_line] = []
            comment_started = False

    
    probs = defaultdict(list)
    for key, value in token_probs.items():
        L = len(token_probs[key])
        if L > 0:
            normalized_prob = math.prod([p ** (1 / L) for p in value])
            probs[key] = normalized_prob
            # print(f"Line [{key}]: {lines[key]} \n {normalized_prob}")
    return lines, probs

In [12]:
max_context_length = model.config.max_position_embeddings

def get_generated_code(prompt, programming_language, comment_symbol):
    inputs = tokenizer(prompt, return_tensors="pt", padding=True)#.to(model.device)
    input_ids = inputs['input_ids']
    context_length = input_ids.shape[1]
    if context_length >= max_context_length:
        print(f"The prompt is too long: {context_length}; Maximum: {max_context_length}")
        return None

    if tokenizer.pad_token_id is None:
        tokenizer.pad_token_id = tokenizer.eos_token_id
    outputs = model.generate(**inputs, 
                             max_length= max_context_length-1, 
                             output_scores=True,
                             output_logits = True,
                             return_dict_in_generate=True)
    generated_tokens = outputs.sequences[0]
    generated_answer = tokenizer.decode(generated_tokens, skip_special_tokens=True)
    
    logits = outputs.logits
    generated_code = get_code_from_text(generated_answer, programming_language).strip()
    return generated_code, get_normalized_prob_line_by_line(logits, generated_tokens, context_length, generated_code, comment_symbol, programming_language, tokenizer)

In [13]:
easy_test = get_generated_code("Can you write a Python function that computes the factorial of a number? Do not add any extra text, explanations or comments.", Language.PYTHON, '#')
print(easy_test[0])
print()
print(easy_test[1][0])
print(easy_test[1][1])

def factorial(n):
    if n == 0:
        return 1
    else:
        return n * factorial(n-1)

defaultdict(<class 'list'>, {0: ' ```python\n', 1: 'def factorial(n):\n', 2: '    if n == 0:\n', 3: '        return 1\n', 4: '    else:\n', 5: '        return n * factorial(n-1)\n'})
defaultdict(<class 'list'>, {0: 0.9833444678383888, 1: 0.9534250522154831, 2: 0.955552345687939, 3: 0.9962464336804056, 4: 0.9971035387335285, 5: 0.9501636979563424})


In [14]:
hard_description = """
There's an old, narrow bridge that a group of N travelers wish to cross in the night. The bridge can only support the weight of at most 2 people. Crossers must stay together and use the group’s only flashlight while on the bridge. Traveler i can cross the bridge in Si seconds alone.

Thankfully, the group had the foresight to bring a (very lightweight!) wheelbarrow. Either:

- traveler i can cross the bridge alone in Si seconds, optionally bringing the wheelbarrow, or

- two travelers i and j can both cross in Sj seconds if traveler j rides in the wheelbarrow

Any group crossing the bridge must bring the flashlight. It can be returned to the initial side by the same rules above. Is there a strategy for all travelers to cross the bridge in K seconds?
# Constraints

1 ≤ T ≤ 95
1 ≤ N ≤ 1,000
1 ≤ Si, K ≤ 1,000,000,000

# Input Format

Input begins with an integer T, the number of test cases. Each case begins with a line containing the integers N and K. Then N lines follow, the ith of which contains the integer Si.

# Output Format

For the ith test case, print "Case #i: " followed by "YES" if the travelers can all make it across the bridge within K seconds, or "NO" if they cannot.

# Sample Explanation

Here’s a possible strategy for the first case. Traveler 3 can carry traveler 4 across, and then return alone. Then traveler 2 can carry traveler 3 across, and then return alone. Then traveler 1 can carry traveler 2 across. This takes 5 + 5 + 2 + 2 + 1 = 15 seconds.

In the second case, there is no strategy that gets all 4 travelers across within 4 seconds.

In the third case, both travelers can cross in exactly the 22 allotted seconds if they travel together.

# Sample Input:
6
4 17
1
2
5
10
4 4
1
2
5
10
2 22
22
22
3 1000000000
1000000000
1000000000
1000000000
1 10
12
1 100
12

# Sample Output:
Case #1: YES
Case #2: NO
Case #3: YES
Case #4: NO
Case #5: NO
Case #6: YES

"""

In [15]:
hard_problem_description = """
You are planning a hike in the Peneda-Gerês National Park in the north of Portugal. The park takes its name from two of its highest peaks: Peneda (1340 m) and Gerês (1545 m).

For this problem, the park is modelled as an infinite plane, where each position (x, y), with x, y being integers, has a specific altitude. The altitudes are defined by an n × n matrix h, which repeats periodically across the plane. Specifically, for any integers a, b and 0 ≤ x, y < n, the altitude at (x + an, y + bn) is h[x][y].

When you are at position (x, y), you can move to any of the four adjacent positions: (x, y + 1), (x + 1, y), (x, y − 1), or (x − 1, y). The time required to move between two adjacent positions is 1 + |alt1 − alt2|, where alt1 and alt2 are the altitudes of the current and destination positions, respectively.

Initially, your position is (0,0). Compute the number of distinct positions you can reach within 10^20 seconds. Your answer will be considered correct if its relative error is less than 10^−6.

Input
The first line contains an integer n (2 ≤ n ≤ 20)—the size of the matrix describing the altitudes.

The following n lines contain n integers each. The (j + 1)-th number on the (i + 1)-th of these lines is h[i][j] (0 ≤ h[i][j] ≤ 1545)—the altitude of the position (i, j).

Output
Print the number of distinct positions you can reach within 10^20 seconds. Your answer will be considered correct if its relative error is less than 10^−6.

Examples

Input
2
3 3
3 3

Output
2e+40

Input
3
0 0 0
0 1545 0
0 0 0

Output
2e+40

Input
4
0 1 2 3
5 6 7 4
10 11 8 9
15 12 13 14

Output
1.524986878e+39
"""


In [16]:
hard_test = get_generated_code(hard_description+extra_instructions, Language.PYTHON, '#')
print(hard_test[0])
print()
print(hard_test[1][0])
print(hard_test[1][1])

No code found in the generated output

defaultdict(<class 'list'>, {})
defaultdict(<class 'list'>, {})


In [17]:
hard_description_test = get_generated_code(hard_problem_description+extra_instructions, Language.PYTHON, '#')
print(hard_description_test[0])
print()
print(hard_description_test[1][0])
print(hard_description_test[1][1])

import sys
input = sys.stdin.read
data = input().split()

n = int(data[0])
h = [[0] * n for _ in range(n)]
index = 1
for i in range(n):
    for j in range(n):
        h[i][j] = int(data[index])
        index += 1

def reachable_positions(n, h, max_time):
    def dfs(x, y, time_left, visited):
        if (x, y) in visited:
            return
        visited.add((x, y))
        for dx, dy in [(-1, 0), (1, 0), (0, -1), (0, 1)]:
            nx, ny = x + dx, y + dy
            if 0 <= nx < n and 0 <= ny < n:
                move_time = 1 + abs(h[nx][ny] - h[x][y])
                if move_time <= time_left:
                    dfs(nx, ny, time_left - move_time, visited)

    visited = set()
    dfs(0, 0, max_time, visited)
    return len(visited)

max_time = 10**20
result = reachable_positions(n, h, max_time)
print(f"{result:.6e}")

defaultdict(<class 'list'>, {0: ' Just provide the code.\n```python\n', 1: 'import sys\n', 2: 'input = sys.stdin.read\n', 3: 'data = input().split()\n\n', 4: 'n 

In [18]:
generated_code = get_generated_code(c_sharp_prompt+selected_problems.iloc[1]["description"]+extra_instructions, Language.C_SHARP, '\\')
print(generated_code[0])
print()
print(generated_code[1][0])
print(generated_code[1][1])

using System;

class Program {
    static void Main() {
        string[] input = new string[3];
        for (int i = 0; i < 3; i++) {
            input[i] = Console.ReadLine();
        }

        long n = ParseLong(input[0].Trim());
        long d = ParseLong(input[2].Trim());

        long gcd = GCD(Math.Abs(n), Math.Abs(d));
        n /= gcd;
        d /= gcd;

        if (d == 1) {
            Console.WriteLine(n);
        } else {
            Console.WriteLine($"{n}/{d}");
        }
    }

    static long ParseLong(string s) {
        long result = 0;
        bool isNegative = false;
        int startIndex = 0;

        if (s[0] == '-') {
            isNegative = true;
            startIndex = 1;
        }

        for (int i = startIndex; i < s.Length; i++) {
            result = result * 10 + (s[i] - '0');
        }

        return isNegative ? -result : result;
    }

    static long GCD(long a, long b) {
        while (b != 0) {
            long temp = b;
            b = a % b;

## Code Generation - Selected Problems

In [19]:
def add_token_prob(title, lines, probs, df):
    for key, value in probs.items():
        df.loc[len(df)] = [title, key, lines[key], value]
    

In [20]:
columns = ['title', 'line_number', 'line_text', 'token_prob']
python_token_prob = pd.DataFrame([], columns=columns)
c_harp_token_prob = pd.DataFrame([], columns=columns)
typescript_token_prob = pd.DataFrame([], columns=columns)

In [21]:
def generate_solution(start_prompt, programming_language, prob_df, comment_symbol):
    problems_with_solutions = selected_problems
    for index, row in tqdm(problems_with_solutions.iterrows()):
        code, linesprobs = get_generated_code(start_prompt+row["description"], programming_language, comment_symbol)
        problems_with_solutions.loc[index, "solution"] = code
        add_token_prob(row["title"], linesprobs[0], linesprobs[1], prob_df)
    return problems_with_solutions

In [22]:
c_sharp_solutions = generate_solution(c_sharp_prompt, Language.C_SHARP, c_harp_token_prob, '//')
c_sharp_solutions.to_csv('datasets/selected_problems_c_sharp_14B_final.csv', index=False)
c_harp_token_prob.to_csv('datasets/probabilities_c_sharp_final.csv', index=False)

30it [6:24:40, 769.36s/it]


In [23]:
typescript_solutions = generate_solution(typescript_prompt, Language.TYPESCRIPT, typescript_token_prob, '//')
typescript_solutions.to_csv('datasets/selected_problems_typescript_14B_final.csv', index=False)
typescript_token_prob.to_csv('datasets/probabilities_typescript_final.csv', index=False)

30it [7:18:48, 877.61s/it] 


In [24]:
python_solutions = generate_solution(python_prompt, Language.PYTHON, python_token_prob, '#')
python_solutions.to_csv('datasets/selected_problems_python_14B_final.csv', index=False)
python_token_prob.to_csv('datasets/probabilities_python_final.csv', index=False)

30it [5:22:10, 644.36s/it]


# Delete Comments from Generated Code

In [25]:
python_csv_file = '/datasets/selected_problems_python_14B_final.csv'
c_sharp_csv_file = '/datasets/selected_problems_c_sharp_14B_final.csv'
typescript_csv_file = '/datasets/selected_problems_typescript_14B_final.csv'

def get_generated_code(csv_file):
    csv_full_path = os.getcwd()+csv_file
    if os.path.exists(csv_full_path):
        return pd.read_csv(csv_full_path)
    else:
        print("File not found.")
        return None
    
python_solutions = get_generated_code(python_csv_file)
c_sharp_solutions = get_generated_code(c_sharp_csv_file)
typescript_solutions = get_generated_code(typescript_csv_file)

In [26]:
def remove_comments(code, comment_symbol):
    comment_symbol_escaped = re.escape(comment_symbol)
    code_without_comments = re.sub(f'{comment_symbol_escaped}.*', '', code)
    code_cleaned = re.sub(r'\n\s*\n', '\n', code_without_comments).strip()

    return code_cleaned

code_solution = '''
# Test
def say_hello():  # This function says hello
    print("Hello, world!")  # Print greeting message
'''
cleaned_code = remove_comments(code_solution, '#')
print(cleaned_code)

def say_hello():  
    print("Hello, world!")


In [27]:
python_solutions["solution_no_comments"] = ''
c_sharp_solutions["solution_no_comments"] = ''
typescript_solutions["solution_no_comments"] = ''

In [28]:
def delete_comments(solutions, programming_language):
    comment_symbol = ''
    if programming_language == Language.PYTHON:
        comment_symbol = '#'
    else:
        comment_symbol = '//'

    for index, row in tqdm(solutions.iterrows()):
         solutions.loc[index, 'solution_no_comments'] = remove_comments(row["solution"], comment_symbol)

In [29]:
delete_comments(python_solutions, Language.PYTHON)

30it [00:00, 3179.19it/s]


In [30]:
print(python_solutions[0:1]["solution_no_comments"].to_list()[0])

def read_input():
    import sys
    input = sys.stdin.read
    data = input().split()
    index = 0
    d = int(data[index])
    index += 1
    datasets = []
    for _ in range(d):
        n = int(data[index])
        index += 1
        scores = list(map(int, data[index:index + n]))
        index += n
        datasets.append(scores)
    return datasets
def competition_ranking(scores):
    sorted_scores = sorted([(score, i) for i, score in enumerate(scores)], reverse=True)
    ranks = [0] * len(scores)
    current_rank = 1
    for i, (_, original_index) in enumerate(sorted_scores):
        if i > 0 and sorted_scores[i][0] != sorted_scores[i - 1][0]:
            current_rank = i + 1
        ranks[original_index] = current_rank
    return ranks
def dense_ranking(scores):
    sorted_scores = sorted([(score, i) for i, score in enumerate(scores)], reverse=True)
    ranks = [0] * len(scores)
    current_rank = 1
    for i, (_, original_index) in enumerate(sorted_scores):
        if i > 0 and

In [31]:
delete_comments(typescript_solutions, Language.TYPESCRIPT)
delete_comments(c_sharp_solutions, Language.C_SHARP)

30it [00:00, 3827.27it/s]
30it [00:00, 4930.80it/s]


In [32]:
c_sharp_solutions.to_csv('datasets/selected_problems_c_sharp_14B_no_comments_final.csv', index=False)
typescript_solutions.to_csv('datasets/selected_problems_typescript_14B_no_comments_final.csv', index=False)
python_solutions.to_csv('datasets/selected_problems_python_14B_no_comments_final.csv', index=False)